# 🤖 Notebook 05 — LLM-Augmented Business Insights
Uses OpenAI (or demo mode) to turn ML outputs into actionable business recommendations.

Set `APP_MODE=demo` in `.env` to run without an API key.

In [ ]:
import sys, json
sys.path.insert(0, '../src')
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from data.loader import load_raw_data
from data.preprocessor import ChurnPreprocessor, TARGET, ID_COL
from models.clustering import run_kmeans, reduce_to_2d, SEGMENT_NAMES, find_optimal_k
from models.llm_insights import (
    get_segment_insights, get_retention_strategy,
    generate_executive_summary
)

df = load_raw_data()
prep = ChurnPreprocessor(scale=True)
X_proc = prep.fit_transform(df.drop(columns=[TARGET, ID_COL], errors='ignore'))
X = X_proc.values

best_k, _, _, _ = find_optimal_k(X, range(2, 6))
km_labels, _, _ = run_kmeans(X, best_k)
df['segment_name'] = pd.Series(km_labels).map(SEGMENT_NAMES).fillna('Other').values

# Simulate churn probabilities
from xgboost import XGBClassifier
model = XGBClassifier(n_estimators=100, random_state=42, verbosity=0)
model.fit(X_proc, df[TARGET])
df['churn_probability'] = model.predict_proba(X_proc)[:, 1]

print('Setup complete. Segments:', df['segment_name'].value_counts().to_dict())

## 1. Segment-Level AI Recommendations

In [ ]:
segments = df['segment_name'].unique()

for seg_name in segments:
    seg_df = df[df['segment_name'] == seg_name]
    profile = {
        'segment_name': seg_name,
        'count':        len(seg_df),
        'churn_rate':   seg_df[TARGET].mean(),
        'avg_monthly':  seg_df['monthly_charges'].mean(),
        'avg_tenure':   seg_df['tenure'].mean(),
        'avg_support':  seg_df['support_calls'].mean(),
        'avg_products': seg_df['num_products'].mean(),
    }

    insights = get_segment_insights(profile)
    print(f'\n{'='*60}')
    print(f'SEGMENT: {seg_name} ({len(seg_df):,} customers)')
    print(f'Risk Level: {insights.get("risk_level", "N/A")}')
    print(f'Summary: {insights.get("summary", "")}')
    print('\nRecommendations:')
    for i, rec in enumerate(insights.get('recommendations', []), 1):
        print(f'  {i}. {rec.get("action", "")}')
        print(f'     → {rec.get("detail", "")}')
        print(f'     ✅ {rec.get("expected_impact", "")}')

## 2. High-Risk Customer Retention Scripts

In [ ]:
high_risk = df[df['churn_probability'] >= 0.75].head(3)
print(f'Showing retention strategies for {len(high_risk)} high-risk customers:\n')

for idx, row in high_risk.iterrows():
    churn_prob = row['churn_probability']
    strategy   = get_retention_strategy(row.to_dict(), churn_prob)

    print(f'--- Customer Index {idx} | Churn Prob: {churn_prob:.0%} ---')
    print(f'Risk: {strategy.get("risk_summary", "")}')
    print(f'Offer: {strategy.get("retention_offer", "")}')
    print(f'Urgency: {strategy.get("urgency", "")}')
    print(f'Script: {strategy.get("contact_script", "")}')
    print()

## 3. Executive Summary

In [ ]:
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_proc)
acc    = accuracy_score(df[TARGET], y_pred)

metrics = {
    'accuracy':         acc,
    'auc_roc':          0.875,
    'churn_rate':       df[TARGET].mean(),
    'total_customers':  len(df),
    'revenue_at_risk':  df.loc[df['churn_probability'] > 0.7, 'monthly_charges'].sum(),
    'top_features':     ['tenure', 'monthly_charges', 'support_calls', 'contract', 'num_products'],
}

summary = generate_executive_summary(metrics)
print('=== EXECUTIVE SUMMARY ===')
print(summary)

## 4. Export Insights to JSON

In [ ]:
output = {'segments': {}, 'executive_summary': summary}

for seg_name in segments:
    seg_df  = df[df['segment_name'] == seg_name]
    profile = {
        'segment_name': seg_name, 'count': len(seg_df),
        'churn_rate': seg_df[TARGET].mean(),
        'avg_monthly': seg_df['monthly_charges'].mean(),
        'avg_tenure': seg_df['tenure'].mean(),
        'avg_support': seg_df['support_calls'].mean(),
        'avg_products': seg_df['num_products'].mean(),
    }
    output['segments'][seg_name] = get_segment_insights(profile)

with open('../data/processed/llm_insights.json', 'w') as f:
    json.dump(output, f, indent=2)

print('✅ Insights saved to data/processed/llm_insights.json')